In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
# tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

# input_ids = tokenizer("Hello, I'm a language model", return_tensors="pt").to(model.device)

# output = model.generate(**input_ids, cache_implementation="static")
# print(tokenizer.decode(output[0], skip_special_tokens=True))

c:\Users\jarre\OneDrive\Documents\GitHub\ICS435Assignment4\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import Dataset
import pandas as pd

df = Dataset.from_pandas(pd.read_csv('data.csv')).remove_columns('ID')
df = df.train_test_split(test_size=.2, shuffle=True
                        #  , seed=42
                        )
train = df['train']
test = df['test']

In [15]:
df['train'].shape

(1297, 1)

In [16]:
df['test'].shape

(325, 1)

In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer, TrainingArguments

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token 
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5913.32it/s]


In [4]:
def tokenize(batch):
    return tokenizer(batch['Joke'], truncation=True, padding=True)

In [5]:
tokenized_train = train.map(tokenize, batched=True, remove_columns=['Joke'])
tokenized_test = test.map(tokenize, batched=True, remove_columns=['Joke'])

Map: 100%|██████████| 325/325 [00:00<00:00, 12664.55 examples/s]


In [6]:
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

blocked_tokenized_train = tokenized_train.map(group_texts, batched=True)
blocked_tokenized_test = tokenized_test.map(group_texts, batched=True)

Map: 100%|██████████| 325/325 [00:00<00:00, 9198.04 examples/s]


In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-jokes",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_strategy="epoch",
    logging_steps=10,
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=blocked_tokenized_train,
    eval_dataset=blocked_tokenized_test,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.460416,6.257886
2,0.506988,6.315427
3,0.370792,6.389376
4,0.424491,6.440617
5,0.344487,6.472299
6,0.392180,6.509925
7,0.322581,6.544817
8,0.318544,6.621268
9,0.284745,6.625659
10,0.273717,6.658259


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]


TrainOutput(global_step=36800, training_loss=0.19941062204215837, metrics={'train_runtime': 2438.164, 'train_samples_per_second': 30.146, 'train_steps_per_second': 15.093, 'total_flos': 4801241088000000.0, 'train_loss': 0.19941062204215837, 'epoch': 100.0})

In [13]:
input_ids = tokenizer("what did the", return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static")
print(tokenizer.decode(output[0], skip_special_tokens=True))

c:\Users\jarre\OneDrive\Documents\GitHub\ICS435Assignment4\.env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=23) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


what did the slab of meat say when it was covered in salt and left out to dry? "I'm cured


In [14]:
input_ids = tokenizer("So this guy", return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static")
print(tokenizer.decode(output[0], skip_special_tokens=True))

So this guy uses a cacti-lator! *This one's not coming back to me...* So
